# Three-Panel Plot: Skymodel | CASA Observation | Fit Residual
Plots zoomed versions of (1) the input skymodel, (2) the CASA pbcor image, and (3) the imfit residual image side by side.

## Configuration

In [1]:
# ── Paths ──────────────────────────────────────────────────────────────────────
PBCOR_PATH     = r'C:\Users\nachi\Documents\JupyterNotebooksLinux\Cosmic Origins\casa_main\pbcor_imgs'
SKYMODEL_PATH  = r'C:\Users\nachi\Documents\JupyterNotebooksLinux\Cosmic Origins\casa_main\skymodels'
RESIDUAL_PATH  = r'C:\Users\nachi\Documents\JupyterNotebooksLinux\Cosmic Origins\casa_main\residual_imgs'  # folder where residual FITS files are saved
RESULTS_PATH   = r'C:\Users\nachi\Documents\JupyterNotebooksLinux\Cosmic Origins\casa_main\fitting_results.json'
MODEL_PATH     = r'C:\Users\nachi\Documents\JupyterNotebooksLinux\Cosmic Origins\casa_main\model_imgs'  # folder where model FITS files are saved
OUT_DIR        = 'plots_three_panel'

# ── Display ────────────────────────────────────────────────────────────────────
CMAP      = 'jet'
LOG_SCALE = False
VMIN_PCT  = 0
VMAX_PCT  = 99.5
DPI       = 130

## Imports

In [ ]:
import os, json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from matplotlib.colors import LogNorm, TwoSlopeNorm
from astropy.io import fits

from plotting_utils import (
    DISTANCES_PC,
    get_pixel_scale_arcsec,
    make_norm,
    make_residual_norm,
    zoom_bounds,
    add_AU_ticks,
    draw_ellipse_on_ax,
    style_ax,
    add_colorbar,
    plot_three_panel,
    plot_three_panel_stack,
)

%matplotlib inline

if OUT_DIR:
    os.makedirs(OUT_DIR, exist_ok=True)


## Load fitting results

In [3]:
with open(RESULTS_PATH, 'r') as f:
    master_dict = json.load(f)

field_names = ['Orion', 'Perseus']
print(f'Loaded results for {len(master_dict)} snapshot(s):')
for snap, fields in master_dict.items():
    for field, axes in fields.items():
        if field not in field_names:
            continue
        for axis in axes:
            print(f'  snapshot {snap} | {field} | axis {axis}')

Loaded results for 23 snapshot(s):
  snapshot 170 | Orion | axis x
  snapshot 170 | Orion | axis y
  snapshot 170 | Orion | axis z
  snapshot 170 | Perseus | axis x
  snapshot 170 | Perseus | axis y
  snapshot 170 | Perseus | axis z
  snapshot 171 | Orion | axis x
  snapshot 171 | Orion | axis y
  snapshot 171 | Orion | axis z
  snapshot 171 | Perseus | axis x
  snapshot 171 | Perseus | axis y
  snapshot 171 | Perseus | axis z
  snapshot 172 | Orion | axis x
  snapshot 172 | Orion | axis y
  snapshot 172 | Orion | axis z
  snapshot 172 | Perseus | axis x
  snapshot 172 | Perseus | axis y
  snapshot 172 | Perseus | axis z
  snapshot 173 | Orion | axis x
  snapshot 173 | Orion | axis y
  snapshot 173 | Orion | axis z
  snapshot 173 | Perseus | axis x
  snapshot 173 | Perseus | axis y
  snapshot 173 | Perseus | axis z
  snapshot 174 | Orion | axis x
  snapshot 174 | Orion | axis y
  snapshot 174 | Orion | axis z
  snapshot 174 | Perseus | axis x
  snapshot 174 | Perseus | axis y
  snapsho

In [4]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import h5py
import yt
import os
import json
%matplotlib inline

def load_dict(path):
    if os.path.exists(path):
        with open(path, 'r') as f:
            return json.load(f)
    print(f"No file found at {path}, starting fresh.")
    return {}

def save_dict(d, path):
    with open(path, 'w') as f:
        json.dump(d, f, indent=2)
    print(f"Saved to {path}")

fitting_path = r'C:\Users\nachi\Documents\JupyterNotebooksLinux\Cosmic Origins\casa_main\fitting_results.json'
master_dict = load_dict(fitting_path) 

mass_path = "mass_results.json"
mass_dict = load_dict(mass_path)

snapshots = master_dict.keys()
field_names = ["Orion","Perseus"]
rows = []
for snapshot in master_dict.keys():
    print(f"{snapshot}")
    snapshot_time = master_dict[snapshot]["time"]
    try:
        masses = mass_dict[f'snapshot_{snapshot}']
    except KeyError:
        print(f"No mass data found for snapshot {snapshot}")
        continue
    for field in master_dict[snapshot].keys():
        if field not in field_names:
            continue
        for axis in master_dict[snapshot][field].keys():
            d = master_dict[snapshot][field][axis]
            rows.append({
                'snapshot':          snapshot,
                'axis':              axis,
                'field':             field,
                'inclination_deg':   d['inc'],
                'mass_fit_Msun':     d['dust_mass_Msun'],
                'true_mass_1e7':     masses[1],
                'true_mass_1e8':     masses[2],
                'true_mass_1e9':     masses[3],
                'pct_diff_1e8':    100 * (masses[2] - d['dust_mass_Msun']) / masses[2],
                'pct_diff_1e9':    100 * (masses[3] - d['dust_mass_Msun']) / masses[3],
                'fitting_radius_AU':    d['radius_AU'],
                'fitting_radius_Tobin':     d['radius_AU_Tobin'],
                'snr': d['snr'],
                "peak_residual_fraction": d['peak_residual_fraction'],
                "time_Myr": snapshot_time
            })

df = pd.DataFrame(rows)


170
171
172
173
174
175
180
118
315
325
327
No mass data found for snapshot 327
330
334
343
386
389
408
160
161
162
163
164
165


In [7]:
plt.rcParams.update({
    'axes.labelsize': 11,
    'axes.titlesize': 11,
    'xtick.labelsize': 30,
    'ytick.labelsize': 30,
    'legend.fontsize': 3,
    'figure.titlesize': 11
})

## Plot a single snapshot

In [ ]:
SNAPSHOT    = '172'
FIELD       = 'Orion'
AXIS        = 'z'
ZOOM_FACTOR = 3

pbcor_fname = f'ALMA_snapshot_{SNAPSHOT}_axis_{AXIS}_{FIELD}_sim_observed_pbcor.fits'
pbcor_fpath = os.path.join(PBCOR_PATH, pbcor_fname)
fit         = master_dict[SNAPSHOT][FIELD][AXIS]

plot_three_panel(pbcor_fpath, fit, SKYMODEL_PATH, RESIDUAL_PATH,
                  zoom_factor=ZOOM_FACTOR, cmap=CMAP, dpi=DPI, out_dir=OUT_DIR, show=True)


In [ ]:
plot_three_panel_stack(
    snapshots=[str(s) for s in range(170, 176)],
    field='Orion',
    axis='x',
    results=master_dict,
    pbcor_dir=PBCOR_PATH,
    skymodel_dir=SKYMODEL_PATH,
    residual_dir=RESIDUAL_PATH,
    zoom_factor=3,
    cmap=CMAP,
    vmin_pct=VMIN_PCT,
    vmax_pct=VMAX_PCT,
    log_scale=LOG_SCALE,
    dpi=DPI,
    out_dir=OUT_DIR,
    show=True,
    mass_dict=mass_dict,
    df=df,
)
